# Train-Test Split
```In this exercise you will experience with an important and often neglected issue in the data scientist work - the train-test split. For a specific dataset we will examine different ways to split it and will understand the limitations and constraints we have to take when creating a good train-test split.```

```~ Ittai Haran```

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
%matplotlib inline


def maps(func, lister): # if you hate python 3 as much as I do you might want to use this
    return list(map(func, lister))

```First, load the dataset. Notice that the dataset is made out of pairs of objects, where each row has the id of each object and the features related to it. How many different objects are there?
We would like to describe the objects and the data using a specific data structure. What structure can best describe the objects and the relations between them (what two objects happen to be in the same sample)?```

In [ ]:
df = pd.read_csv('data.csv', index_col=0)
df.head()

Answer: Best structure is a graph, we use a df just to load it from the .csv

```Use networkx to create a graph describing the objects and the relations between them. How many connected components the graph has? Draw a histogram of their sizes. Are there any edges between left objects and right objects? That kind of graph is called a bipartite graph. For any graph computations, networkx is your friend, and it should be very easy.```

In [ ]:
graph = nx.from_pandas_edgelist(df, source="index_left", target="index_right", edge_attr="target")
concomp = [c for c in nx.connected_components(graph)]
plt.hist([len(c) for c in concomp])

```In order to get a baseline model we will try to have predictions using only one object from each sample. Create a dataset containing only the left objects. Drop duplicates, so every object will appear only once.```

In [ ]:
df_left = df.drop(columns=["index_right"] + [f"feature_{i}_right" for i in range(1, 101)]).drop_duplicates()

```Split your data randomly with ratio 0.7-0.3. Train a simple model (a random forest, maybe?) to predict the target. Make sure your model isn't overfitted, and try to get the best score you can (on the test segment). Compare your results to a simple baseline - the mean of the target computed on the train segment.```

In [ ]:
inputs = df_left.drop(columns=["target", "index_left"])
target = df_left["target"]
def train_test_baseline(inputs, target):
    x_train, x_test, y_train, y_test = train_test_split(inputs, target, test_size=0.3)
    clf = RandomForestRegressor(max_depth=16).fit(x_train, y_train)
    mean = np.mean(y_train)
    baseline_y_predict = np.full(y_test.shape, mean)
    baseline_mse = mean_squared_error(y_test, baseline_y_predict)
    test_mse = mean_squared_error(y_test, clf.predict(x_test))
    train_mse = mean_squared_error(y_train, clf.predict(x_train))
    print(f"Baseline MSE: {baseline_mse}\nTest MSE: {test_mse}\nTrain MSE: {train_mse}")
train_test_baseline(inputs, target)


```Repeat that process, only this use all of the sample, and not just the left object. Accordingly, you don't have to drop any duplicates. Use the naive train-test split. Did you get a good score on both train and test? why (or why not)? Do you think the score you got on the test corresponds to the "real" generalization error? why, or why not?```

In [ ]:
inputs = df.drop(columns=["target", "index_left", "index_right"])
target = df["target"]
train_test_baseline(inputs, target)


answer: test score can be great because the test set isnt really seperate from the train set - both contain the same nodes and so the samples (=edges) are far from independent. In particular the test error doesnt correspond well to the actual generalization error, which would entail taking nodes that were truly never seen befroe, i.e. nodes with no edges in the training set

```We will now create a new train-test split, so that every connected component is contained either in the train segment or in the test segment. To do so, implement the following algorithm:```

```while length(train_segment)<0.7*length(data):
    choose randomly a sample s from the data (that is not in train_segment)
    add the connected component containing s to the train_segment
test_segment = data - train_segment```

In [ ]:
train_segment = pd.DataFrame()
test_segment = df.copy()
train_part = 0.7
train_size = train_part * len(df)
connected = []
for c in concomp:
    connected.append(df.loc[(df["index_left"].isin(c)) & (df["index_right"].isin(c))])
train_segment_parts = [] 
current_train_size = 0
not_used = pd.Series(True, index=df.index)


while current_train_size < train_size:
    s = df[not_used].sample()
    for i, c in enumerate(concomp):
        if s["index_left"].iloc[0] in c:
            train_segment_parts.append(connected[i])
            current_train_size += len(connected[i])
            not_used[connected[i].index] = False
            break

train_segment = pd.concat(train_segment_parts)
both = pd.merge(df, train_segment[["index_left", "index_right"]], how="outer", on=["index_left", "index_right"], indicator=True)
test_segment = both[both["_merge"] == "left_only"].drop(columns="_merge")
test_segment.head()    

A way to print the data:


In [ ]:
rand_gen = np.random.default_rng()
def visualize(segment, cutoff):
    big_graph = nx.from_pandas_edgelist(segment, source="index_left",
                                       target="index_right", edge_attr="target")
    shortened_nodes = rand_gen.choice(
        list(big_graph.nodes),
        size=cutoff,
        replace=False
    )
    shortened_graph = big_graph.subgraph(shortened_nodes).copy()
    layout = nx.spring_layout(shortened_graph, weight="target", seed=0)
    nx.draw(shortened_graph, layout,
         node_size=20, edge_color=[d["target"] for *_,d in shortened_graph.edges(data=True)],
         with_labels=False)
    plt.show()

visualize(train_segment, 5000)
visualize(test_segment, 5000)





```Train a good model using your train segment. What is the best score you can get on your test? What is the problem with the train-test split method we used? Hint: How many connected components are there in the train segment, and how many are in the test segment? Examine also the distribution of the target, both in the train and in the test. Do they look the same?```

In [ ]:
def xy_from_train_test(train_segment, test_segment):
    x_train = train_segment.drop(columns=["index_left", "index_right", "target"])
    x_test = test_segment.drop(columns=["index_left", "index_right", "target"])
    y_train = train_segment["target"]
    y_test = test_segment["target"]
    return x_train, x_test, y_train, y_test


def fit_eval(clf, x_train, x_test, y_train, y_test):
    clf.fit(x_train, y_train)
    test_mse = mean_squared_error(y_test, clf.predict(x_test))
    train_mse = mean_squared_error(y_train, clf.predict(x_train))
    print(f"Test MSE: {test_mse}\n Train MSE: {train_mse}")
    return test_mse, train_mse

x_train, x_test, y_train, y_test = xy_from_train_test(train_segment, test_segment)
plt.figure()
plt.hist(y_train)
plt.figure()
plt.hist(y_test)


depth_test_mses = [] 
depth_train_mses = [] 

for d in range(1, 21):
    clf = RandomForestRegressor(max_depth=d)
    test_mse, train_mse = fit_eval(clf, x_train, x_test, y_train, y_test)
    depth_test_mses.append(test_mse)
    depth_train_mses.append(train_mse)

print(np.argmin(depth_test_mses) + 1)
print(np.argmin(depth_train_mses) + 1)

plt.figure()
plt.scatter(range(1, 21), depth_test_mses)
plt.scatter(range(1, 21), depth_train_mses)

best_depth = np.argmin(depth_test_mses) + 1





There is large variance in the sizes of the components, we might end up with very few components in the "70%" training set so the learning wont be effective.


```Do the train-test split again, only this time make sure you have ~0.7 of the connected components in your train segment, using a different algorithm.```

In [ ]:
def build_df_from_ccs(concomps):
    connected = []
    for c in concomps:
        connected.append(df.loc[(df["index_left"].isin(c)) & (df["index_right"].isin(c))])
    segment = pd.concat(connected)
    return segment


rand_gen = np.random.default_rng(9)


training_indices = rand_gen.choice(np.arange(len(concomp)), size=int(train_part*len(concomp)), replace=False)
test_indices = np.array([i for i in np.arange(len(concomp)) if i not in training_indices])


training_concomp = [concomp[i] for i in training_indices]
testing_concomp = [concomp[i] for i in test_indices]

train_segment = build_df_from_ccs(training_concomp)
test_segment = build_df_from_ccs(testing_concomp)

x_train, x_test, y_train, y_test = xy_from_train_test(train_segment, test_segment)
clf = RandomForestRegressor(max_depth=best_depth)
fit_eval(clf, x_train, x_test, y_train, y_test)





Making sure the split maintains the 70% logic:

In [ ]:
used = [False for _ in concomp]
num_comps = 0
for index, s in train_segment.iterrows(): # type: ignore
    for i, c in enumerate(concomp):
        if not used[i] and s["index_left"] in c:
            num_comps += 1
            used[i] = True
ratio = num_comps / len(concomp)
print(f"{num_comps} components in train_segment, {len(concomp)} in entire dataset\n", f"ratio: {ratio}")


```What part of the connected components you have in your train segment this time? Try also look again at the distribution of the target in the two segments.```

In [ ]:
print(int(train_part*len(concomp))/len(concomp))
plt.figure()
plt.hist(y_train)
plt.figure()
plt.hist(y_test)

```Train a good model using you train segment. What is the best score you can get on your test? Did you get a better score? why?```

Score is better since we train on more components

Bonus: the data for this exercise was uniquely generated, using MNIST (what? how???). Generating toy datasets is a very useful tool for a data scientist - it can help test hypotheses in small scale or extreme cases. 

How would you approach the problem of creating the dataset for this exercise (either on MNIST or based on another dataset)? Think of the following questions, write the answers and be prepared to discuss them with your instructor:
1. What "features" will the dataset be made of? Remember we want a dataset which uses pairs of objects
2. How will the label be defined?
3. How would you choose which pairs to sample for the dataset? Think for instance if you wanted to create the case, decribed in the exercise, where some of the connected components are very big - how would you do this? What parameters impact this?

1+2. We can cluster the original data and draw edges between samples that end up in the same cluster, with the label/edge weight being their distance.
3. we can adjust the clustering algorithm, for instance with DBSCAN we'd use bigger $\varepsilon\text{s}$ to get bigger connected components. some other parameters that could affec this are how many samples we take and different parameters in other algorithms (e.g. k in KNN)